In [35]:
import pandas as pd 
import nannyml as nml

reference_df = pd.read_csv('reference.csv')
analysis_df = pd.read_csv('analysis.csv')
nml.disable_usage_logging()

reference_df.head()



,timestamp,time_since_login_min,transaction_amount,transaction_type,is_first_transaction,user_tenure_months,is_fraud,predicted_fraud_proba,predicted_fraud
0,2018-01-01 00:00:00.000,1.561750,3981.1,PAYMENT,False,0.318980,1.0,0.99,1
1,2018-01-01 00:08:43.152,1.658074,1267.9,PAYMENT,False,7.391323,0.0,0.07,0
2,2018-01-01 00:17:26.304,2.454287,1984.7,CASH-IN,False,0.781225,1.0,1.00,1
3,2018-01-01 00:26:09.456,2.392085,2265.2,CASH-OUT,False,0.680473,1.0,0.98,1
4,2018-01-01 00:34:52.608,2.189806,2126.8,CASH-IN,False,8.542895,1.0,0.99,1


In [36]:
reference_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50207 entries, 0 to 50206
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   timestamp              50207 non-null  str    
 1   time_since_login_min   50207 non-null  float64
 2   transaction_amount     50207 non-null  float64
 3   transaction_type       47155 non-null  str    
 4   is_first_transaction   50207 non-null  bool   
 5   user_tenure_months     50207 non-null  float64
 6   is_fraud               50207 non-null  float64
 7   predicted_fraud_proba  50207 non-null  float64
 8   predicted_fraud        50207 non-null  int64  
dtypes: bool(1), float64(5), int64(1), str(2)
memory usage: 3.1 MB


In [37]:
#Configuration de l'estimateur de performance CBPE de NannyML

estimator= nml.CBPE(
    problem_type='classification_binary',
    y_pred_proba= 'predicted_fraud_proba',
    y_pred='predicted_fraud',
    y_true='is_fraud',
    metrics=['accuracy'],
    timestamp_column_name= 'timestamp',
    chunk_period ='M'  # Pour le decoupage mensuel
)


# Ajustement de l'estimateur sur le dataset de reference et estimer sur le dataset d'analyse
estimator.fit(reference_df)
results =estimator.estimate(analysis_df)

# Recupération du dataframe des résultats
results_df= results.to_df()

# 4. Filtrer les blocs mensuels déclenchant une alerte
# On extrait les lignes où la colonne d'alerte de l'accuracy vaut True

alert_rows = results_df[results_df[('accuracy','alert')]==True]

#Formatage du nom des mois 

months_with_performance_alerts = [
    f"{row[('chunk', 'start_date')].strftime('%B').lower()}_{row[('chunk', 'start_date')].year}"
    for _, row in alert_rows.iterrows()
]

print("months_with_performance_alerts =", months_with_performance_alerts)


# Dictionnaire de traduction des mois du français vers l'anglais
fr_to_en = {
    'janvier': 'january', 'février': 'february', 'mars': 'march', 
    'avril': 'april', 'mai': 'may', 'juin': 'june', 
    'juillet': 'july', 'août': 'august', 'septembre': 'september', 
    'octobre': 'october', 'novembre': 'november', 'décembre': 'december'
}

months_with_performance_alerts = []
for _, row in alert_rows.iterrows():
    # Conversion de la date
    start_date = pd.to_datetime(row[('chunk', 'start_date')])
    month_name = start_date.strftime('%B').lower()
    
    # Traduction si la date sort en français
    month_name_en = fr_to_en.get(month_name, month_name)
    
    months_with_performance_alerts.append(f"{month_name_en}_{start_date.year}")

print("months_with_performance_alerts =", months_with_performance_alerts)

months_with_performance_alerts = ['avril_2019', 'mai_2019', 'juin_2019']
months_with_performance_alerts = ['april_2019', 'may_2019', 'june_2019']


In [38]:
# 1. Identifier la liste des features/variables indépendantes
features = [
    'time_since_login_min', 
    'transaction_amount', 
    'transaction_type', 
    'is_first_transaction', 
    'user_tenure_months'
]

# 2. Calculer la dérive univariée pour ces features
calc = nml.UnivariateDriftCalculator(
    column_names=features,
    timestamp_column_name='timestamp',
    chunk_period='M'
)

calc.fit(reference_df)
drift_results = calc.calculate(analysis_df)

# 3. Calculer la corrélation entre la dérive des features et la baisse de performance
# NannyML propose un module de correlation / rank_features ou on peut analyser les résultats
# Alternative direct NannyML :
# On cherche quelle feature est la plus corrélée avec la baisse d'accuracy dans results
# En pratique, calculons les scores de dérive par feature
drift_df = drift_results.to_df()

# On peut aussi corréler les résultats de dérive avec la performance estimée :
# Ou si NannyML possède la méthode Correlation / Ranker :
# Pour être certain du nom exact :
# Observons les colonnes ou calculons le niveau de dérive moyen par feature





In [39]:
import numpy as np

# 1. Extraire les résultats sous forme de DataFrame Pandas
drift_df = drift_results.to_df()
res_df = results.to_df()

# 2. Récupérer la série des valeurs d'accuracy
accuracy_values = res_df[('accuracy', 'value')].values

correlations = {}

for col in features:
    # On cherche la colonne correspondant à la valeur de dérive (value) pour la feature
    if (col, 'value') in drift_df.columns:
        drift_values = drift_df[(col, 'value')].values
    else:
        # Fallback pour les colonnes plates
        matching_cols = [c for c in drift_df.columns if col in str(c) and 'value' in str(c)]
        drift_values = drift_df[matching_cols[0]].values
    
    # Calcul de la corrélation de Pearson (en valeur absolue)
    corr = np.corrcoef(drift_values, accuracy_values)[0, 1]
    correlations[col] = abs(corr)

# 3. Récupérer la caractéristique avec la plus forte corrélation
highest_correlation_feature = max(correlations, key=correlations.get)

print("highest_correlation_feature =", highest_correlation_feature)

highest_correlation_feature = time_since_login_min


/tmp/ipykernel_8849/3981639980.py:14: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (col, 'value') in drift_df.columns:
/tmp/ipykernel_8849/3981639980.py:14: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (col, 'value') in drift_df.columns:


In [55]:
# --- TÂCHE 3 : Montant moyen de transaction lors de l'alerte ---
calc = nml.UnivariateDriftCalculator(
    column_names=['transaction_amount'],
    timestamp_column_name='timestamp',
    chunk_period='M'
)
calc.fit(reference_df)
drift_results = calc.calculate(analysis_df)
drift_df = drift_results.to_df()

# 1. Isoler le mois (chunk) en alerte
alert_cols = [c for c in drift_df.columns if 'alert' in str(c)]
start_col = [c for c in drift_df.columns if 'start_date' in str(c)][0]
end_col = [c for c in drift_df.columns if 'end_date' in str(c)][0]

alert_chunk = drift_df[drift_df[alert_cols].any(axis=1)]
start_date = alert_chunk[start_col].values[0]
end_date = alert_chunk[end_col].values[0]

# 2. Calculer le montant moyen des transactions sur juin 2019
analysis_df['timestamp'] = pd.to_datetime(analysis_df['timestamp'])
alert_period_df = analysis_df[
    (analysis_df['timestamp'] >= start_date) & 
    (analysis_df['timestamp'] <= end_date)
]

alert_avg_transaction_amount = float(alert_period_df['transaction_amount'].mean())

print("alert_avg_transaction_amount =", alert_avg_transaction_amount)

alert_avg_transaction_amount = 3069.8183551967713
